In [1]:
!pip install -q handyspark findspark

In [2]:
import os
import numpy as np
if not hasattr(np, "bool"):
    np.bool = np.bool_   # redirect to the correct scalar type
import findspark
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from handyspark import *
from matplotlib import pyplot as plt
# fixes issue with seaborn hiding fliers on boxplot
import matplotlib as mpl
mpl.rc("lines", markeredgewidth=0.5)

findspark.init()
os.environ['PYSPARK_SUBMIT_ARGS'] = '--master local[*] pyspark-shell'

spark = SparkSession.builder \
    .appName("InstaCartEDA").enableHiveSupport().getOrCreate()

%matplotlib inline

/tmp/ipykernel_780/287089385.py:3: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "bool"):


In [3]:
aisles = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/aisles.csv", header=True, inferSchema=True)
departments = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/departments.csv", header=True, inferSchema=True)
products = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/orders.csv", header=True, inferSchema=True)
order_products_prior = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/order_products__prior.csv", header=True, inferSchema=True)
order_products_train = spark.read.csv("hdfs://hadoop-namenode:9000/user/project/data/raw/order_products__train.csv", header=True, inferSchema=True)

In [4]:
def null_report(df):
    """
    Returns a Spark DataFrame with null counts per column.
    Includes: true nulls, empty strings, 'NULL', 'None', and NaN.
    """
    return df.select([
        F.count(
            F.when(
                F.col(c).isNull() | 
                (F.col(c) == '') | 
                (F.col(c) == 'NULL') | 
                (F.col(c) == 'None') | 
                F.isnan(F.col(c)), 
                c
            )
        ).alias(c)
        for c in df.columns
    ])

In [5]:
dfs_dict = {
    "aisles": aisles,
    "departments": departments,
    "products": products,
    "orders": orders,
    "order_products_prior": order_products_prior,
    "order_products_train": order_products_train
}
for name, df in dfs_dict.items():
    print(f'Shape of the dataset: {name}', (df.count(), len(df.columns)))
    df.show(n=10)
    df.printSchema()

Shape of the dataset: aisles (134, 2)
+--------+--------------------+
|aisle_id|               aisle|
+--------+--------------------+
|       1|prepared soups sa...|
|       2|   specialty cheeses|
|       3| energy granola bars|
|       4|       instant foods|
|       5|marinades meat pr...|
|       6|               other|
|       7|       packaged meat|
|       8|     bakery desserts|
|       9|         pasta sauce|
|      10|    kitchen supplies|
+--------+--------------------+
only showing top 10 rows

root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)

Shape of the dataset: departments (21, 2)
+-------------+---------------+
|department_id|     department|
+-------------+---------------+
|            1|         frozen|
|            2|          other|
|            3|         bakery|
|            4|        produce|
|            5|        alcohol|
|            6|  international|
|            7|      beverages|
|            8|           pets|
|         

### First Problem Found: 
#### aisle_id, department_id are both string in products

In [6]:
# Null Exploration
print("Looking for Null Values ...")
for name, df in dfs_dict.items():
    print(f"Table: {name}")
    null_report(df).show()

Looking for Null Values ...
Table: aisles
+--------+-----+
|aisle_id|aisle|
+--------+-----+
|       0|    0|
+--------+-----+

Table: departments
+-------------+----------+
|department_id|department|
+-------------+----------+
|            0|         0|
+-------------+----------+

Table: products
+----------+------------+--------+-------------+
|product_id|product_name|aisle_id|department_id|
+----------+------------+--------+-------------+
|         0|           0|       0|            0|
+----------+------------+--------+-------------+

Table: orders
+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                206209|
+--------+-------+--------+------------+---------+-----------------+-----

### Second Problem Found:
#### Null Values found in 'orders', col: 'days_since_prior_order'

In [7]:
for title, df in dfs_dict.items():
    print(f"Table: {title}")
    df.describe().show()


Table: aisles
+-------+-----------------+--------------------+
|summary|         aisle_id|               aisle|
+-------+-----------------+--------------------+
|  count|              134|                 134|
|   mean|             67.5|                NULL|
| stddev|38.82653731663435|                NULL|
|    min|                1|air fresheners ca...|
|    max|              134|              yogurt|
+-------+-----------------+--------------------+

Table: departments
+-------+------------------+----------+
|summary|     department_id|department|
+-------+------------------+----------+
|  count|                21|        21|
|   mean|              11.0|      NULL|
| stddev|6.2048368229954285|      NULL|
|    min|                 1|   alcohol|
|    max|                21|    snacks|
+-------+------------------+----------+

Table: products
+-------+------------------+--------------------+-----------------+------------------+
|summary|        product_id|        product_name|         ais

### Cleaning and Processing Checklist:
#### - Data Integrity in 'products'
#### - Null Values in 'orders' solving 
#### - Orders missing in 'order_products_prior' (Still in Development)
#### - Products names needs cleaning in 'products'
#### - Checking for duplicates and outliers
#### - Checking for correct join procedure after cleaning 
#### - Saving the Data 

In [8]:
products.select("department_id").distinct().show()
products.select("aisle_id").distinct().show()

+-------------+
|department_id|
+-------------+
|            7|
|           15|
|           11|
|            3|
|            8|
|           16|
|            5|
|           18|
|           17|
|         Red"|
|            6|
|           19|
|            9|
|            1|
|           20|
|           10|
|            4|
|           12|
|           13|
|           14|
+-------------+
only showing top 20 rows

+--------+
|aisle_id|
+--------+
|     125|
|       7|
|      51|
|     124|
|      15|
|      54|
|     132|
|      11|
|     101|
|      69|
|      29|
|     112|
|      42|
|      87|
|      73|
|      64|
|       3|
| Blunted|
|      30|
|     113|
+--------+
only showing top 20 rows



In [9]:
products.filter(~products.department_id.cast("int").isNotNull()).show(truncate=False)
products.filter(~products.aisle_id.cast("int").isNotNull()).show(truncate=False)

+----------+-------------------------+--------+-------------+
|product_id|product_name             |aisle_id|department_id|
+----------+-------------------------+--------+-------------+
|6816      |"Scotch Kids 5"" Scissors| Blunted| Red"        |
+----------+-------------------------+--------+-------------+

+----------+-------------------------+--------+-------------+
|product_id|product_name             |aisle_id|department_id|
+----------+-------------------------+--------+-------------+
|6816      |"Scotch Kids 5"" Scissors| Blunted| Red"        |
+----------+-------------------------+--------+-------------+



In [10]:
products = products.filter(~products.product_id.isin(6816))

In [11]:
products.select("department_id").distinct().show()
products.select("aisle_id").distinct().show()

+-------------+
|department_id|
+-------------+
|            7|
|           15|
|           11|
|            3|
|            8|
|           16|
|            5|
|           18|
|           17|
|            6|
|           19|
|            9|
|            1|
|           20|
|           10|
|            4|
|           12|
|           13|
|           14|
|           21|
+-------------+
only showing top 20 rows

+--------+
|aisle_id|
+--------+
|     125|
|       7|
|      51|
|     124|
|      15|
|      54|
|     132|
|      11|
|     101|
|      69|
|      29|
|     112|
|      42|
|      87|
|      73|
|      64|
|       3|
|      30|
|     113|
|      34|
+--------+
only showing top 20 rows



In [12]:
products.columns

['product_id', 'product_name', 'aisle_id', 'department_id']

In [13]:
from pyspark.sql.types import IntegerType
colum = []
for c in products.columns:
    if c in ['department_id', 'aisle_id']:
        colum.append(products[c].cast(IntegerType()).alias(c))
    else:
        colum.append(products[c])
    

In [14]:
products = products.select(colum)
products.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)



First Problem Solved!

In [15]:
orders.columns

['order_id',
 'user_id',
 'eval_set',
 'order_number',
 'order_dow',
 'order_hour_of_day',
 'days_since_prior_order']

In [16]:
orders = orders.na.fill(value=0, subset=["days_since_prior_order"])
orders.select([
    F.count(
        F.when(
            F.col(c).isNull() | 
            (F.col(c) == '') | 
            (F.col(c) == 'NULL') | 
            (F.col(c) == 'None') | 
            F.isnan(F.col(c)), 
            c
        )
    ).alias(c) 
        for c in orders.columns]).show()

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                     0|
+--------+-------+--------+------------+---------+-----------------+----------------------+



In [19]:
for title, df in dfs_dict.items():
    print(f"Table: {title}")
    df.describe().show()

Table: aisles
+-------+-----------------+--------------------+
|summary|         aisle_id|               aisle|
+-------+-----------------+--------------------+
|  count|              134|                 134|
|   mean|             67.5|                NULL|
| stddev|38.82653731663435|                NULL|
|    min|                1|air fresheners ca...|
|    max|              134|              yogurt|
+-------+-----------------+--------------------+

Table: departments
+-------+------------------+----------+
|summary|     department_id|department|
+-------+------------------+----------+
|  count|                21|        21|
|   mean|              11.0|      NULL|
| stddev|6.2048368229954285|      NULL|
|    min|                 1|   alcohol|
|    max|                21|    snacks|
+-------+------------------+----------+

Table: products
+-------+------------------+--------------------+-----------------+------------------+
|summary|        product_id|        product_name|         ais

In [21]:
from pyspark.sql.functions import regexp_replace, trim, lower

# Clean product names (remove quotes, trim spaces)
products_clean = products.withColumn(
    "product_name",
    trim(regexp_replace("product_name", '"', ""))
)

# Normalize aisle and department names
aisles_clean = aisles.withColumn(
    "aisle",
    lower(trim(regexp_replace("aisle", '"', "")))
)

departments_clean = departments.withColumn(
    "department",
    lower(trim(regexp_replace("department", '"', "")))
)


In [26]:
products_clean.describe().show()
aisles_clean.describe().show()

+-------+------------------+--------------------+-----------------+------------------+
|summary|        product_id|        product_name|         aisle_id|     department_id|
+-------+------------------+--------------------+-----------------+------------------+
|  count|             49687|               49687|            49687|             49687|
|   mean|24844.862841387083|              1493.0|67.76919516171232|11.728580916537524|
| stddev|14343.750735683467|   620.4329133758138|38.31645086851397| 5.850420587359203|
|    min|                 1|   #2 Coffee Filters|                1|                 1|
|    max|             49688|with a Splash of ...|              134|                21|
+-------+------------------+--------------------+-----------------+------------------+

+-------+-----------------+--------------------+
|summary|         aisle_id|               aisle|
+-------+-----------------+--------------------+
|  count|              134|                 134|
|   mean|           

In [22]:
from pyspark.sql.functions import when

# Impute nulls in days_since_prior_order with 0 (first order)
orders_clean = orders.withColumn(
    "days_since_prior_order",
    when(orders["days_since_prior_order"].isNull(), 0)
    .otherwise(orders["days_since_prior_order"])
)


In [31]:
orders_clean.describe().show()

+-------+-----------------+------------------+--------+------------------+------------------+-----------------+----------------------+
|summary|         order_id|           user_id|eval_set|      order_number|         order_dow|order_hour_of_day|days_since_prior_order|
+-------+-----------------+------------------+--------+------------------+------------------+-----------------+----------------------+
|  count|          3421083|           3421083| 3421083|           3421083|           3421083|          3421083|               3421083|
|   mean|        1710542.0|102978.20805926077|    NULL|17.154857979183785|2.7762191095626734|13.45201534134074|     10.44487900468945|
| stddev|987581.7398225744|59533.717793502125|    NULL|17.733164470966702|2.0468291939879775|4.226088402102001|      9.30872656199968|
|    min|                1|                 1|   prior|                 1|                 0|                0|                   0.0|
|    max|          3421083|            206209|   train|

In [23]:
# Check for products with aisle_id not in aisles
invalid_aisles = products_clean.join(
    aisles_clean, "aisle_id", "left_anti"
)
invalid_aisles.show()

# Check for products with department_id not in departments
invalid_departments = products_clean.join(
    departments_clean, "department_id", "left_anti"
)
invalid_departments.show()


+--------+----------+------------+-------------+
|aisle_id|product_id|product_name|department_id|
+--------+----------+------------+-------------+
+--------+----------+------------+-------------+

+-------------+----------+------------+--------+
|department_id|product_id|product_name|aisle_id|
+-------------+----------+------------+--------+
+-------------+----------+------------+--------+



In [30]:
# Orders missing in order_products_prior
missing_in_prior = orders_clean.join(
    order_products_prior, "order_id", "left_anti"
)
print(missing_in_prior.count())
# Orders missing in order_products_train
missing_in_train = orders_clean.join(
    order_products_train, "order_id", "left_anti"
)
print(missing_in_train.count())


206209
3289874


In [35]:
order_details = (order_products_prior
    .join(products_clean, "product_id")
    .join(aisles_clean, "aisle_id")
    .join(departments_clean, "department_id")
    .join(orders_clean, "order_id"))

In [36]:
print("----- Table: order details -----")
order_details.show(10)
order_details.printSchema()

----- Table: order details -----
+--------+-------------+--------+----------+-----------------+---------+--------------------+--------------------+---------------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|department_id|aisle_id|product_id|add_to_cart_order|reordered|        product_name|               aisle|     department|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------------+--------+----------+-----------------+---------+--------------------+--------------------+---------------+-------+--------+------------+---------+-----------------+----------------------+
|      31|            7|      94|     48946|                1|        1|       Earl Grey Tea|                 tea|      beverages| 201744|   prior|           7|        6|               15|                  14.0|
|      31|            4|      83|     22935|                2|        1|Organic Yellow Onion|    fresh vegetables|     

In [37]:
order_details.describe().show()

+-------+------------------+-----------------+------------------+------------------+------------------+-------------------+--------------------+--------------------+----------+------------------+--------+------------------+-----------------+------------------+----------------------+
|summary|          order_id|    department_id|          aisle_id|        product_id| add_to_cart_order|          reordered|        product_name|               aisle|department|           user_id|eval_set|      order_number|        order_dow| order_hour_of_day|days_since_prior_order|
+-------+------------------+-----------------+------------------+------------------+------------------+-------------------+--------------------+--------------------+----------+------------------+--------+------------------+-----------------+------------------+----------------------+
|  count|          32434486|         32434486|          32434486|          32434486|          32434486|           32434486|            32434486|    

In [38]:
# Detect duplicates in order_details
duplicates = (order_details
    .groupBy(order_details.columns)   # group by all columns
    .count()
    .filter("count > 1")              # keep only rows that appear more than once
)
print(f"Number of duplicate rows: {duplicates.count()}")


Number of duplicate rows: 0


In [39]:
from pyspark.sql.functions import col
from pyspark.sql.types import NumericType

def tukey_outliers_all(df, k=1.5):
    """
    Detect Tukey outliers for every numeric column in a DataFrame.
    Returns a dictionary with column name -> outlier count.
    """
    outlier_summary = {}
    numeric_cols = [c for c, t in df.dtypes if isinstance(df.schema[c].dataType, NumericType)]
    
    for c in numeric_cols:
        q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
        iqr = q3 - q1
        lower, upper = q1 - k * iqr, q3 + k * iqr
        
        # Count outliers
        outlier_count = df.filter((col(c) < lower) | (col(c) > upper)).count()
        outlier_summary[c] = {
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "LowerBound": lower,
            "UpperBound": upper,
            "OutlierCount": outlier_count
        }
    
    return outlier_summary


In [40]:
summary = tukey_outliers_all(order_details, k=1.5)
for col_name, stats in summary.items():
    print(f"{col_name}: {stats}")

order_id: {'Q1': 842761.0, 'Q3': 2548496.0, 'IQR': 1705735.0, 'LowerBound': -1715841.5, 'UpperBound': 5107098.5, 'OutlierCount': 0}
department_id: {'Q1': 4.0, 'Q3': 16.0, 'IQR': 12.0, 'LowerBound': -14.0, 'UpperBound': 34.0, 'OutlierCount': 0}
aisle_id: {'Q1': 31.0, 'Q3': 107.0, 'IQR': 76.0, 'LowerBound': -83.0, 'UpperBound': 221.0, 'OutlierCount': 0}
product_id: {'Q1': 13649.0, 'Q3': 37766.0, 'IQR': 24117.0, 'LowerBound': -22526.5, 'UpperBound': 73941.5, 'OutlierCount': 0}
add_to_cart_order: {'Q1': 3.0, 'Q3': 12.0, 'IQR': 9.0, 'LowerBound': -10.5, 'UpperBound': 25.5, 'OutlierCount': 997175}
reordered: {'Q1': 0.0, 'Q3': 1.0, 'IQR': 1.0, 'LowerBound': -1.5, 'UpperBound': 2.5, 'OutlierCount': 0}
user_id: {'Q1': 51524.0, 'Q3': 154158.0, 'IQR': 102634.0, 'LowerBound': -102427.0, 'UpperBound': 308109.0, 'OutlierCount': 0}
order_number: {'Q1': 5.0, 'Q3': 24.0, 'IQR': 19.0, 'LowerBound': -23.5, 'UpperBound': 52.5, 'OutlierCount': 1783828}
order_dow: {'Q1': 1.0, 'Q3': 5.0, 'IQR': 4.0, 'LowerBo

In [49]:
# Save order_products_train
order_products_train.write.mode("overwrite").parquet("hdfs://hadoop-namenode:9000/user/project/data/preprocessed/order_products_train")

# Save order_products_prior
order_products_prior.write.mode("overwrite").parquet("hdfs://hadoop-namenode:9000/user/project/data/preprocessed/order_products_prior")

# Save products
products_clean.write.mode("overwrite").parquet("hdfs://hadoop-namenode:9000/user/project/data/preprocessed/products")

# Save aisles
aisles_clean.write.mode("overwrite").parquet("hdfs://hadoop-namenode:9000/user/project/data/preprocessed/aisles")

# Save departments
departments_clean.write.mode("overwrite").parquet("hdfs://hadoop-namenode:9000/user/project/data/preprocessed/departments")

# Save orders
orders_clean.write.mode("overwrite").parquet("hdfs://hadoop-namenode:9000/user/project/data/preprocessed/orders")


Py4JJavaError: An error occurred while calling o1759.parquet.
: org.apache.hadoop.security.AccessControlException: Permission denied: user=jovyan, access=WRITE, inode="/user/project/data/preprocessed":root:supergroup:drwxr-xr-x
	at org.apache.hadoop.hdfs.server.namenode.FSPermissionChecker.check(FSPermissionChecker.java:399)
	at org.apache.hadoop.hdfs.server.namenode.FSPermissionChecker.checkPermission(FSPermissionChecker.java:255)
	at org.apache.hadoop.hdfs.server.namenode.FSPermissionChecker.checkPermission(FSPermissionChecker.java:193)
	at org.apache.hadoop.hdfs.server.namenode.FSDirectory.checkPermission(FSDirectory.java:1879)
	at org.apache.hadoop.hdfs.server.namenode.FSDirectory.checkPermission(FSDirectory.java:1863)
	at org.apache.hadoop.hdfs.server.namenode.FSDirectory.checkAncestorAccess(FSDirectory.java:1822)
	at org.apache.hadoop.hdfs.server.namenode.FSDirMkdirOp.mkdirs(FSDirMkdirOp.java:59)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.mkdirs(FSNamesystem.java:323

In [41]:
from pyspark.sql.functions import col, count, avg, sum as _sum

# --- Order-level features ---
order_features = (order_products_prior
    .groupBy("order_id")
    .agg(count("product_id").alias("basket_size"))
)

# --- Product-level features ---
product_features = (order_products_prior
    .groupBy("product_id")
    .agg(
        count("order_id").alias("product_count"),
        avg("reordered").alias("product_reorder_rate")
    )
)

# --- User-level features ---
user_features = (orders_clean
    .groupBy("user_id")
    .agg(
        avg("order_number").alias("avg_order_number"),
        avg("days_since_prior_order").alias("avg_days_between_orders")
    )
)

user_reorder_ratio = (order_products_prior
    .join(orders_clean, "order_id")
    .groupBy("user_id")
    .agg(avg("reordered").alias("user_reorder_ratio"))
)

# Combine user features
user_features = user_features.join(user_reorder_ratio, "user_id", "left")

# --- Final training dataset ---
train_df = (order_products_train
    .join(products_clean, "product_id")
    .join(aisles_clean, "aisle_id")
    .join(departments_clean, "department_id")
    .join(orders_clean, "order_id")
    .join(order_features, "order_id")
    .join(product_features, "product_id")
    .join(user_features, "user_id")
)


In [47]:
train_df = (order_products_train
    .join(products_clean, "product_id")              # adds aisle_id, department_id
    .join(aisles_clean, "aisle_id")                  # now aisle_id exists
    .join(departments_clean, "department_id")        # now department_id exists
    .join(orders_clean, "order_id")                  # adds user_id, order info
    .join(order_features, "order_id")
    .join(product_features, "product_id")
    .join(user_features, "user_id")
)

train_df.show(10)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [44]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Index categorical features
aisle_indexer = StringIndexer(inputCol="aisle", outputCol="aisle_index", handleInvalid="keep")
dept_indexer = StringIndexer(inputCol="department", outputCol="dept_index", handleInvalid="keep")

# One-hot encode
aisle_encoder = OneHotEncoder(inputCol="aisle_index", outputCol="aisle_vec")
dept_encoder = OneHotEncoder(inputCol="dept_index", outputCol="dept_vec")

# Assemble features (no product_id)
assembler = VectorAssembler(
    inputCols=[
        "add_to_cart_order",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
        "aisle_vec",
        "dept_vec"
    ],
    outputCol="features"
)

# GBT Classifier
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="reordered",
    maxIter=20,
    maxDepth=5
)

# Pipeline
pipeline = Pipeline(stages=[aisle_indexer, dept_indexer, aisle_encoder, dept_encoder, assembler, gbt])

# Train/test split
train, test = train_df.randomSplit([0.8, 0.2], seed=42)

# Fit model
model = pipeline.fit(train)

# Evaluate
predictions = model.transform(test)
evaluator = BinaryClassificationEvaluator(labelCol="reordered", metricName="areaUnderROC")
print("Test AUC:", evaluator.evaluate(predictions))


IllegalArgumentException: requirement failed: The input column aisle_index should have at least two distinct values.